In [ ]:
import sys
sys.path.append("/opt/app-root/src/drive/My Libraries/Cluster_Prob")
sys.path.append("/opt/app-root/src/brainscales2-demos.git")

In [ ]:
from _static.common.helpers import setup_hardware_client, get_nightly_calibration
setup_hardware_client()

import copy
import json
from pathlib import Path
import matplotlib.pyplot as plt
import network
import os
import sys
import time
import pickle
import socket
import inspect
import hashlib
import platform
from datetime import datetime, timezone

import numpy as np
import pynn_brainscales.brainscales2 as sim
from pynn_brainscales.brainscales2 import helper as bs2_helper
import pynn_brainscales.brainscales2 as sim

from matplotlib.font_manager import FontProperties

def scale_fontsize(value, factor=1.2):
    # convert string sizes like "large" to numeric
    size = FontProperties(size=value).get_size_in_points()
    return size * factor

plt.rcParams.update({
    "font.size": scale_fontsize(plt.rcParams["font.size"]),
    "axes.titlesize": scale_fontsize(plt.rcParams["axes.titlesize"]),
    "axes.labelsize": scale_fontsize(plt.rcParams["axes.labelsize"]),
    "xtick.labelsize": scale_fontsize(plt.rcParams["xtick.labelsize"]),
    "ytick.labelsize": scale_fontsize(plt.rcParams["ytick.labelsize"]),
})

In [ ]:
try:
    import matplotlib
except Exception:
    matplotlib = None


def _safe_attr(obj, name, default=None):
    try:
        return getattr(obj, name)
    except Exception:
        return default


def _safe_call(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception:
        return None


def _safe_repr(obj):
    try:
        return repr(obj)
    except Exception as exc:
        return f"<repr failed: {exc}>"


def _safe_version(dist_name, fallback_module=None):
    try:
        from importlib.metadata import version
        return version(dist_name)
    except Exception:
        return (
            getattr(fallback_module, "__version__", None)
            if fallback_module is not None
            else None
        )


def _to_pickleable(obj):
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj

    if isinstance(obj, np.generic):
        return obj.item()

    if isinstance(obj, np.ndarray):
        return obj.copy()

    if isinstance(obj, dict):
        return {str(k): _to_pickleable(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple)):
        return [_to_pickleable(v) for v in obj]

    try:
        pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL)
        return obj
    except Exception:
        return _safe_repr(obj)


# Backward-compatible alias for the first snippet.
_make_pickleable = _to_pickleable


def _get_connection_uid(conn):
    if conn is None:
        return None

    try:
        return conn.get_unique_identifier()
    except TypeError:
        try:
            return conn.get_unique_identifier(None)
        except Exception:
            return None
    except Exception:
        return None


def _get_network_info(network_module=None):
    """
    Returns:
        network_source, network_sha256, network_file
    """
    if network_module is None:
        network_module = globals().get("network")

    if network_module is None:
        return None, None, None

    try:
        network_source = inspect.getsource(network_module.ClusteredNetwork)
    except Exception:
        network_source = None

    network_sha256 = (
        hashlib.sha256(network_source.encode("utf-8")).hexdigest()
        if network_source is not None
        else None
    )

    network_file = getattr(network_module, "__file__", None)

    return network_source, network_sha256, network_file


def _population_hwparams(ei_network):
    out = {}

    for pops in ei_network._populations:
        for pop in pops:
            out[pop.label] = {
                "size": int(pop.size),
                "actual_hwparams": _safe_repr(
                    _safe_attr(pop, "actual_hwparams")
                ),
                "calib_hwparams": _safe_repr(
                    _safe_attr(pop, "calib_hwparams")
                ),
                "calib_target": _safe_repr(
                    _safe_attr(pop, "calib_target")
                ),
            }

    return out


def _population_placements(ei_network, state):
    placement = _safe_attr(state, "neuron_placement")

    if placement is None:
        return None

    out = {}

    for pops in ei_network._populations:
        for pop in pops:
            try:
                coords = placement.id2first_circuit(list(pop.all_cells))
                out[pop.label] = [int(c) for c in coords]
            except Exception:
                out[pop.label] = None

    return out


def _synapse_dac_bias(chip_cfg):
    if chip_cfg is None or not hasattr(chip_cfg, "synapse_blocks"):
        return None

    try:
        return [
            [int(v) for v in block.i_bias_dac]
            for block in chip_cfg.synapse_blocks
        ]
    except Exception:
        return _safe_repr(chip_cfg.synapse_blocks)


def _hardware_snapshot(ei_network, calib=None):
    state = sim.simulator.state
    conn = _safe_attr(state, "conn")
    chip_cfg = _safe_attr(state, "grenade_chip_config")

    execution_time_info = _safe_call(sim.get_execution_time_info)
    if execution_time_info is None:
        execution_time_info = _safe_attr(state, "execution_time_info")

    return {
        "connection_unique_identifier": _get_connection_uid(conn),
        "chip_unique_identifier": _safe_call(bs2_helper.get_unique_identifier),

        "connection_is_quiggeldy": (
            _safe_call(getattr(conn, "is_quiggeldy", lambda: None))
            if conn is not None
            else None
        ),
        "connection_bitfile_info": (
            _safe_call(getattr(conn, "get_bitfile_info", lambda: None))
            if conn is not None
            else None
        ),
        "connection_remote_repo_state": (
            _safe_call(getattr(conn, "get_remote_repo_state", lambda: None))
            if conn is not None
            else None
        ),
        "connection_time_info": (
            _safe_repr(_safe_call(getattr(conn, "get_time_info", lambda: None)))
            if conn is not None
            else None
        ),

        "backend_statistics": _safe_repr(
            _safe_call(sim.get_backend_statistics)
        ),
        "execution_time_info": _safe_repr(execution_time_info),
        "initial_config": _safe_repr(_safe_attr(state, "initial_config")),
        "nightly_calibration": _safe_repr(calib),
        "grenade_chip_config": _safe_repr(chip_cfg),
        "neuron_placement": _safe_repr(_safe_attr(state, "neuron_placement")),
        "population_placements": _population_placements(ei_network, state),
        "population_hwparams": _population_hwparams(ei_network),
        "synapse_dac_bias": _synapse_dac_bias(chip_cfg),
        "n_projections": len(getattr(ei_network, "_projections", [])),
        "projection_labels": [
            getattr(p, "label", None)
            for p in getattr(ei_network, "_projections", [])
        ],
    }


def _build_payload(
    ei_network,
    result,
    kappa,
    run_id,
    calib,
    t0_utc,
    t1_utc,
    elapsed_s,
    network_module=None,
):
    network_source, network_sha256, network_file = _get_network_info(
        network_module
    )

    return {
        "identifier": run_id,
        "kappa": float(kappa),
        "spiketimes": result["spiketimes"],
        "e_rate_hz": float(result["e_rate"]),
        "i_rate_hz": float(result["i_rate"]),
        "params": _to_pickleable(result.get("_params")),
        "run_metadata": {
            "utc_start": t0_utc.isoformat(),
            "utc_end": t1_utc.isoformat(),
            "elapsed_seconds": float(elapsed_s),
            "hostname": socket.gethostname(),
            "platform": platform.platform(),
            "python": sys.version,
            "cwd": os.getcwd(),
            "network_module_file": network_file,
            "network_clusterednetwork_sha256": network_sha256,
            "network_clusterednetwork_source": network_source,
            "versions": {
                "numpy": np.__version__,
                "matplotlib": (
                    matplotlib.__version__
                    if matplotlib is not None
                    else None
                ),
                "pynn_brainscales": _safe_version(
                    "pynn-brainscales",
                    sim,
                ),
                "pyNN": _safe_version("PyNN"),
            },
        },
        "hardware_metadata": _hardware_snapshot(ei_network, calib),
    }


def _collect_extended_info(
    *,
    ei_network,
    result,
    sim_dict_run,
    net_dict_run,
    stim_dict_run,
    t0_utc,
    t1_utc,
    wall_t0,
    clusterednetwork_source=None,
    clusterednetwork_sha256=None,
    network_module=None,
):
    if clusterednetwork_source is None or clusterednetwork_sha256 is None:
        network_source, network_sha256, network_file = _get_network_info(
            network_module
        )

        if clusterednetwork_source is None:
            clusterednetwork_source = network_source

        if clusterednetwork_sha256 is None:
            clusterednetwork_sha256 = network_sha256
    else:
        _, _, network_file = _get_network_info(network_module)

    return {
        "net_dict": _to_pickleable(net_dict_run),
        "sim_dict": _to_pickleable(sim_dict_run),
        "stim_dict": _to_pickleable(stim_dict_run),
        "merged_params": _to_pickleable(result.get("_params")),
        "run_metadata": {
            "utc_start": t0_utc.isoformat(),
            "utc_end": t1_utc.isoformat(),
            "elapsed_seconds": float(time.time() - wall_t0),
            "hostname": socket.gethostname(),
            "platform": platform.platform(),
            "python": sys.version,
            "cwd": os.getcwd(),
            "network_module_file": network_file,
            "clusterednetwork_sha256": clusterednetwork_sha256,
            "clusterednetwork_source": clusterednetwork_source,
            "randseed": sim_dict_run.get("randseed"),
        },
        "hardware_metadata": _hardware_snapshot(
            ei_network,
            calib=sim_dict_run.get("initial_config"),
        ),
    }

In [ ]:
net_dict = {
    ############################################
    # neuron parameters
    ############################################
    # neuron model
    "neuron_type": "iaf_psc_exp",
    # Resting potential [mV]
    "E_L": 0.0,
    # Membrane capacitance [pF]
    "C_m": 1.0,
    # Membrane time constant for excitatory neurons [ms]
    "tau_E": 20.0,
    # Membrane time constant for inhibitory neurons [ms]
    "tau_I": 10.0,
    # Refractory period [ms]
    "t_ref": 5.0,
    # Threshold for excitatory neurons [mV]
    "V_th_E": 20.0,
    # Threshold for inhibitory neurons [mV]
    "V_th_I": 20.0,
    # Reset potential [mV]
    "V_r": 0.0,
    # synaptic time constant for excitatory synapses [ms]
    "tau_syn_ex": 5.0,
    # synaptic time constant for inhibitory synapses [ms]
    "tau_syn_in": 5.0,
    # synaptic delay [ms]
    "delay": 0.1,
    # Feed forward excitatory input [rheobase current]
    "I_th_E": 1.05,
    # Feed forward inhibitory input [rheobase current]
    "I_th_I": 0.95,
    # distribution of feed forward input,
    # I_th*[1-delta_I_../2, 1+delta_I_../2]
    "delta_I_xE": 0.0,  # excitatory
    "delta_I_xI": 0.0,  # inhibitory
    # initial membrane potential: either a float (in mV) to initialize all neurons to a fixed value
    # or "rand" for randomized values: "V_th_{E,I}" - 20 * nest.random.lognormal(0, 1)
    "V_m": "rand",
    ############################################
    # network parameters
    ############################################
    # number of excitatory neurons in the network
    # Neurons per cluster N_E/n_clusters
    "N_E": 240,
    # number of inhibitory neurons in the network
    "N_I": 60,
    # Number of clusters
    "n_clusters": 6,
    # connection probabilities
    # baseline_conn_prob[0, 0] E to E, baseline_conn_prob[0, 1] I to E,
    # baseline_conn_prob[1, 0] E to I, baseline_conn_prob[1, 1] I to I
    "baseline_conn_prob": 0.15*np.ones((2,2)),
    # inhibitory weight ratios - scaling like random balanced network
    "gei": 1.2,  # I to E
    "gie": 1.0,  # E to I
    "gii": 1.0,  # I to I
    # additional scaling factor for all weights
    # - can be used to scale weights with network size
    "s": 1.0,
    # fixed indegree - otherwise established with probability ps
    "fixed_indegree": True,
    # cluster network by "weight" or "probabilities"
    "kappa": 0.0,
    # ratio excitatory to inhibitory clustering,
    # rj = 0 means no clustering, which resembles a clustered network
    # with a blanket of inhibition
    "rj": 0.65,
    # excitatory clustering factor,
    # rep = 1 means no clustering, reselmbles a balanced random network
    "rep": 4.5,
}


stim_dict = {
    # list of clusters to be stimulated (None: no stimulation, 0-n_clusters-1)
    "stim_clusters": None,#[2, 3, 4],
    # stimulus amplitude (in pA)
    "stim_amp": 0.00015,
    # stimulus start times in ms: list (warmup time is added automatically)
    "stim_starts": [2000, 6000],
    # list of stimulus end times in ms (warmup time is added automatically)
    "stim_ends": [3500, 7500],
}

sim_dict = {"warmup": 80.0,
            "simtime": 100.0,
            "dt": 0.1,
            "randseed": 1,
            "n_vp": 4,
            "initial_config": get_nightly_calibration()}

KAPPAS = [0.0, 0.5, 1.0]   # change these if you want different three values
OUTDIR = Path("bs2_cluster_runs")
OUTDIR.mkdir(exist_ok=True)

In [ ]:
manifest = []
try:
    clusterednetwork_source = inspect.getsource(network.ClusteredNetwork)
    clusterednetwork_sha256 = hashlib.sha256(
        clusterednetwork_source.encode("utf-8")
    ).hexdigest()
except Exception:
    clusterednetwork_source = None
    clusterednetwork_sha256 = None

for kappa in KAPPAS:
    run_id = f"bs2_cluster_kappa_{kappa:.2f}".replace(".", "p")
    outfile = OUTDIR / f"{run_id}.pkl"
    rasterfile = OUTDIR / f"{run_id}_raster.png"

    net_dict_run = copy.deepcopy(net_dict)
    stim_dict_run = copy.deepcopy(stim_dict)
    sim_dict_run = dict(sim_dict)

    # Keep this if you want every sweep run to explicitly use the nightly calibration.
    # Remove it if sim_dict["initial_config"] was already set upstream and should not be refreshed.
    sim_dict_run["initial_config"] = get_nightly_calibration()

    net_dict_run["kappa"] = float(kappa)
    net_dict_run.pop("clustering", None)

    t0_utc = datetime.now(timezone.utc)
    wall_t0 = time.time()

    try:
        ei_network = network.ClusteredNetwork(
            sim_dict_run,
            net_dict_run,
            stim_dict_run,
        )
        result = ei_network.get_simulation()
        t1_utc = datetime.now(timezone.utc)

        payload = _build_payload(
            ei_network=ei_network,
            result=result,
            kappa=kappa,
            run_id=run_id,
            calib=sim_dict_run["initial_config"],
            t0_utc=t0_utc,
            t1_utc=t1_utc,
            elapsed_s=time.time() - wall_t0,
        )

        payload.update(
            _collect_extended_info(
                ei_network=ei_network,
                result=result,
                sim_dict_run=sim_dict_run,
                net_dict_run=net_dict_run,
                stim_dict_run=stim_dict_run,
                t0_utc=t0_utc,
                t1_utc=t1_utc,
                wall_t0=wall_t0,
                clusterednetwork_source=clusterednetwork_source,
                clusterednetwork_sha256=clusterednetwork_sha256,
            )
        )

        with open(outfile, "wb") as f:
            pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

        manifest.append({
            "identifier": run_id,
            "kappa": float(kappa),
            "pickle_file": str(outfile),
            "raster_file": str(rasterfile),
            "e_rate_hz": float(result["e_rate"]),
            "i_rate_hz": float(result["i_rate"]),
            "utc_start": t0_utc.isoformat(),
            "utc_end": t1_utc.isoformat(),
            "elapsed_seconds": float(time.time() - wall_t0),
            "randseed": sim_dict_run.get("randseed"),
            "chip_unique_identifier": payload["hardware_metadata"]["chip_unique_identifier"],
            "n_projections": payload["hardware_metadata"]["n_projections"],
        })

        print(f"saved {outfile}")
        print(
            f"kappa={kappa:.2f} | "
            f"E={result['e_rate']:.2f} Hz | "
            f"I={result['i_rate']:.2f} Hz"
        )

    finally:
        try:
            sim.end()
        except Exception:
            pass

with open(OUTDIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"saved {OUTDIR / 'manifest.json'}")

In [ ]:
# ------------------------------------------------------------------
# Set your three pickle files here
# ------------------------------------------------------------------
pickle_files = [
    "bs2_cluster_runs/bs2_cluster_kappa_0p00.pkl",
    "bs2_cluster_runs/bs2_cluster_kappa_0p50.pkl",
    "bs2_cluster_runs/bs2_cluster_kappa_1p00.pkl",
]

save_path = "bs2_cluster_runs/kappa_comparison_raster.png"


def load_run(path):
    with open(path, "rb") as f:
        data = pickle.load(f)

    params = data["params"]
    N_E = int(params["N_E"])
    N_I = int(params["N_I"])
    kappa = float(data["kappa"])
    spiketimes = np.asarray(data["spiketimes"])

    return {
        "path": str(path),
        "kappa": kappa,
        "N_E": N_E,
        "N_I": N_I,
        "spiketimes": spiketimes,
    }


runs = [load_run(Path(p)) for p in pickle_files]
runs = sorted(runs, key=lambda d: d["kappa"])

# Common x-range across all panels
nonempty_runs = [run for run in runs if run["spiketimes"].shape[1] > 0]
t_min = min(run["spiketimes"][0].min() for run in nonempty_runs)
t_max = 6#max(run["spiketimes"][0].max() for run in nonempty_runs)



fig, axes = plt.subplots(1, 3, figsize=(13,4), sharex=True, sharey=True)
panel_labels = ["a", "b", "c"]

for i, (ax, run, panel) in enumerate(zip(axes, runs, panel_labels)):
    spk = run["spiketimes"]
    N_E = run["N_E"]
    N_I = run["N_I"]
    kappa = run["kappa"]

    if spk.shape[1] > 0:
        t = spk[0]
        gid = spk[1]

        exc_mask = gid < N_E
        inh_mask = gid >= N_E

        ax.plot(
            t[exc_mask],
            gid[exc_mask],
            ".",
            color="k",
            markersize=1.0,
            rasterized=True,
        )
        ax.plot(
            t[inh_mask],
            gid[inh_mask],
            ".",
            color="darkred",
            markersize=1.0,
            rasterized=True,
        )

    ax.set_xlim(t_min, t_max)
    ax.set_ylim(-1, N_E + N_I)
    ax.set_xlabel(r"t [ms]")   # change to "t[ms]" if your times are in ms
    ax.set_yticks([])
    ax.set_title(rf"$\kappa={kappa:g}$", pad=8)

    if i == 0:
        x0, x1 = ax.get_xlim()
        x_text = x0 - 0.02 * (x1 - x0)
        
        y_exc = 0.5 * (N_E - 1)
        y_inh = N_E + 0.5 * (N_I - 1)
        
        x_text = x0 - 0.02 * (x1 - x0)

        y_exc = 0.5 * (N_E - 1)
        y_inh = N_E + 0.5 * (N_I - 1)

        # normalized positions
        y_exc = (0.5 * (N_E - 1)) / (N_E + N_I)
        y_inh = (N_E + 0.5 * (N_I - 1)) / (N_E + N_I)
        
        ax.text(
            -0.04, y_exc, "exc.",
            transform=ax.transAxes,
            color="k",
            rotation=90,
            va="center", ha="center",
            clip_on=False,
        )
        ax.text(
            -0.04, y_inh, "inh.",
            transform=ax.transAxes,
            color="darkred",
            rotation=90,
            va="center", ha="center",
            clip_on=False,
        )

fig.subplots_adjust(left=0.12, right=0.98, bottom=0.18, top=0.82, wspace=0.08)
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()